# Translation-command dropout: p=0.2

**270 episodes:** three LIBERO tasks × ten initial states × nine methods, all at `p=0.2`.

Methods: fixed `K=1, 5, 10, 25, 50` and DVAC with execution caps `C=5, 10, 25, 50`. At each control step, the clipped XYZ command is replaced by zero with probability 0.2; rotation and gripper commands are preserved. The checkpoint, initial states, policy seed and step-indexed disturbance stream match the other experiment cohorts.

Enable **GPU** and **Internet** in Kaggle, import this notebook, and select **Run All**. No individual-cell editing, repository upload or extra dataset is required. The full code is embedded, including the fixes for system Python without ensurepip and inherited uv install directories. Dependencies, model weights and simulator assets use the frozen project versions.

The other eight methods run first; the expensive fixed `K=1` runs last. Completed episodes are saved individually. A method enters the comparison table only after all of its 30 episodes finish. Missing or unfinished episodes are never counted as failures. The historical `p=0` baseline is reused, with no additional full baseline episodes. Three short excluded checks verify the p=0 wrapper before collection.

Hardware and differences from the historical first prediction are recorded. All nine methods in this notebook share the same Kaggle GPU; elapsed times across different machines must be analyzed separately. Outputs use the separate `action-dropout-kaggle-p020` directory.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, subprocess, sys, zipfile, io

EMBEDDED_BUNDLE = '...'
BUNDLE_BYTES = base64.b64decode(EMBEDDED_BUNDLE)
# A revised bundle gets a fresh setup directory; previous results stay intact.
BUNDLE_ID = hashlib.sha256(BUNDLE_BYTES).hexdigest()[:12]
ROOT = Path(f"/tmp/vla-dropout-kaggle-{BUNDLE_ID}")
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(BUNDLE_BYTES)) as archive:
    for entry in archive.infolist():
        target = (ROOT / entry.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()) or entry.filename.startswith("/"):
            raise RuntimeError("Unsafe embedded archive path")
        if target.exists() and target.is_file() and target.read_bytes() != archive.read(entry):
            raise RuntimeError(f"Existing source differs: {target}; choose a fresh ROOT")
    archive.extractall(ROOT)
manifest = json.loads((ROOT / "bundle-manifest.json").read_text())
for name, expected in manifest["files"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
os.environ["HF_HOME"] = "/tmp/vla-dropout-hf-cache"
os.environ["PIP_CACHE_DIR"] = "/tmp/vla-dropout-pip-cache"
os.environ["UV_CACHE_DIR"] = "/tmp/vla-dropout-uv-cache"
print("Verified embedded bundle:", len(manifest["files"]), "files")
print("Results will be saved under /kaggle/working")


## Install the frozen environment and download public assets

This installs the pinned standalone uv binary, then uses an isolated managed Python 3.12.3 and the exact project dependency lock. It does not require the notebook kernel's pip or ensurepip. Keep Internet enabled for this cell. Subsequent collection uses offline caches. If Kaggle reports an unavailable package or incompatible CUDA driver, stop and retain the error; do not silently replace package versions. See the [official uv installer documentation](https://docs.astral.sh/uv/reference/installer/).


In [ ]:
subprocess.run(["bash", str(ROOT / "scripts/setup_action_dropout_kaggle.sh")], cwd=ROOT, check=True)
PYTHON = ROOT / ".venv/bin/python"
print("Environment ready:", PYTHON)


## Optional resume

For a fresh run leave `RESUME_ZIP` empty. To resume a saved run, attach the previously downloaded `action-dropout-kaggle-p020-results.zip` as a Kaggle input and set its path below. The collector verifies every completed episode and requires the same GPU type, package versions, configuration and source bundle. A different GPU/runtime starts a separate cohort and cannot silently resume these results. Completed episodes in the current session are reused automatically.


In [ ]:
RESUME_ZIP = ""  # e.g. /kaggle/input/my-dropout-checkpoint/action-dropout-kaggle-p020-results.zip
OUTPUT = Path("/kaggle/working/action-dropout-kaggle-p020")
ARCHIVE = Path("/kaggle/working/action-dropout-kaggle-p020-results.zip")
if RESUME_ZIP:
    if OUTPUT.exists():
        raise RuntimeError("Output already exists; do not overwrite a current run with a resume archive")
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        for entry in archive.infolist():
            target = (Path("/kaggle/working") / entry.filename).resolve()
            if not target.is_relative_to(OUTPUT.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise RuntimeError("Unsafe resume archive path or symbolic link")
        archive.extractall("/kaggle/working")
    print("Resume files extracted; all evidence will be verified before reuse.")


## Run all 270 episodes

The default runs the entire registered plan. `MAX_HOURS=10` is a soft limit checked between episodes, excluding setup and the short invariance checks. Each finished episode is saved atomically. A downloadable ZIP is refreshed every five episodes and when the collector exits normally or handles an interruption. A hard session termination may leave only the previous ZIP checkpoint.

The eight methods other than fixed K=1 run first. Progress prints the current method, task, initial state, success, model calls and elapsed time. Only complete 30-episode method panels enter the final table. Already completed episodes are verified and reused if the collection is resumed in the same environment.

Kaggle **Save Version → Save & Run All** starts a separate execution session. Download the ZIP from the active session or save the finished notebook outputs.


In [ ]:
MAX_HOURS = 10.0
EPISODE_LIMIT = None
command = [str(PYTHON), "-u", str(ROOT / "scripts/run_action_dropout_kaggle.py"),
           "--output", str(OUTPUT), "--archive", str(ARCHIVE), "--max-hours", str(MAX_HOURS)]
if EPISODE_LIMIT is not None:
    command += ["--limit", str(EPISODE_LIMIT)]
process = subprocess.Popen(command, cwd=ROOT)
try:
    code = process.wait()
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    process.wait()  # allow atomic export of completed episodes
    raise
if code:
    raise RuntimeError(f"Collector stopped with exit code {code}; inspect the error above and attempts/error.json")


## Verify and download the p=0.2 results

The ZIP includes raw action chunks, commanded and applied actions, disturbance masks, videos, provenance and the short invariance checks. Send `action-dropout-kaggle-p020-results.zip` back for comparison with the p=0, 0.1 and 0.3 results. Partial collection is explicitly reported.


In [ ]:
subprocess.run([str(PYTHON), str(ROOT / "scripts/run_action_dropout_kaggle.py"),
                "--output", str(OUTPUT), "--archive", str(ARCHIVE), "--verify-only"], cwd=ROOT, check=True)
from IPython.display import display, FileLink
progress = json.loads((OUTPUT / "progress.json").read_text())
print(f"Completed: {progress['completed_episodes']}/{progress['planned_episodes']} episodes")
print("Complete methods:", progress.get("complete_methods", []))
for row in progress["conditions"]:
    print(f"{row['method']} | p={row['p']:.1f}: {row['successes']}/{row['episodes']} success; "
          f"mean calls={row['mean_model_calls']:.1f}; realized dropout={row['drop_fraction']:.3f}")
display(FileLink(str(ARCHIVE)))
